# 02 - Data Preprocessing & Feature Engineering

This notebook builds and demonstrates the preprocessing pipeline defined in `training/preprocess.py`, which is reused identically during training (`training/train_model.py`) and at prediction time (`backend/app/predictor.py`). Using the *same* `ColumnTransformer` object in both places prevents training/serving skew.

In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'training'))
import pandas as pd
from preprocess import build_preprocessor, CATEGORICAL_FEATURES, NUMERIC_FEATURES, ALL_FEATURES, TARGET_COLUMN

DATA_PATH = os.path.join('..', 'backend', 'data', 'career_dataset.csv')
df = pd.read_csv(DATA_PATH)
df.shape

(2200, 23)

## Feature groups

In [2]:
print('Categorical features (-> OneHotEncoder):')
print(CATEGORICAL_FEATURES)
print('\nNumeric features (-> StandardScaler):')
print(NUMERIC_FEATURES)

Categorical features (-> OneHotEncoder):
['education_level', 'field_of_study', 'preferred_work_type']

Numeric features (-> StandardScaler):
['years_experience', 'python_level', 'javascript_level', 'java_level', 'sql_level', 'react_level', 'nodejs_level', 'machine_learning_level', 'data_analysis_level', 'cloud_level', 'cybersecurity_level', 'statistics_level', 'communication_level', 'problem_solving_level', 'interest_ai', 'interest_web', 'interest_data', 'interest_cloud', 'interest_security']


## Why OneHotEncoder for categoricals?

`education_level`, `field_of_study`, and `preferred_work_type` are **nominal** categories - there's no meaningful numeric order between 'Remote' and 'On-site'. One-hot encoding avoids implying a false ordering that a model like Logistic Regression could otherwise misinterpret as a magnitude relationship.

## Why StandardScaler for numeric features?

Skill/interest levels are already small ordered integers (0-3), and `years_experience` ranges much wider (0-15+). Scaling puts every numeric feature on a comparable scale, which matters for distance/gradient-sensitive models like Logistic Regression. Tree-based models (Random Forest, Gradient Boosting) don't strictly need this, but applying it uniformly keeps the comparison between models fair and the pipeline simple.

In [3]:
X = df[ALL_FEATURES]
y = df[TARGET_COLUMN]

preprocessor = build_preprocessor()
X_transformed = preprocessor.fit_transform(X)
print('Original shape:', X.shape)
print('Transformed shape (after one-hot + scaling):', X_transformed.shape)

Original shape: (2200, 22)
Transformed shape (after one-hot + scaling): (2200, 34)


In [4]:
feature_names = preprocessor.get_feature_names_out()
print(f'{len(feature_names)} features after preprocessing:')
print(list(feature_names))

34 features after preprocessing:
['categorical__education_level_Bachelor', 'categorical__education_level_Intermediate/High School', 'categorical__education_level_Master', 'categorical__education_level_PhD', 'categorical__field_of_study_Business/Commerce', 'categorical__field_of_study_Computer Science', 'categorical__field_of_study_Data Science', 'categorical__field_of_study_Electrical Engineering', 'categorical__field_of_study_Information Technology', 'categorical__field_of_study_Mathematics', 'categorical__field_of_study_Other', 'categorical__field_of_study_Software Engineering', 'categorical__preferred_work_type_Hybrid', 'categorical__preferred_work_type_On-site', 'categorical__preferred_work_type_Remote', 'numeric__years_experience', 'numeric__python_level', 'numeric__javascript_level', 'numeric__java_level', 'numeric__sql_level', 'numeric__react_level', 'numeric__nodejs_level', 'numeric__machine_learning_level', 'numeric__data_analysis_level', 'numeric__cloud_level', 'numeric__cybe

## Train/test split (stratified)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, ' Test:', X_test.shape)
print('\nTrain class balance:')
print(y_train.value_counts(normalize=True).round(3))

Train: (1760, 22)  Test: (440, 22)

Train class balance:
career
Full Stack Developer         0.1
DevOps Engineer              0.1
Frontend Developer           0.1
AI Engineer                  0.1
Data Scientist               0.1
Cloud Engineer               0.1
Data Analyst                 0.1
Cybersecurity Analyst        0.1
Machine Learning Engineer    0.1
Backend Developer            0.1
Name: proportion, dtype: float64


## Summary

- The `ColumnTransformer` cleanly separates categorical vs numeric handling.
- One-hot encoding + scaling together produce a fully numeric matrix the model can train on.
- A stratified 80/20 split preserves class balance in both sets.
- Next step: `03_model_training.ipynb` trains and compares models on this preprocessed data.